In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_quiver(
    ax,
    x, y, dx, dy,
    color,
    title,
    step=8,
    scale=30,
    normalize=True,
    cmap="viridis"
):
    idx = np.arange(0, len(x), step)

    X, Y = x[idx], y[idx]
    U, V = dx[idx], dy[idx]
    C    = color[idx]

    if normalize:
        speed = np.hypot(U, V) + 1e-12
        U = U / speed
        V = V / speed

    q = ax.quiver(
        X, Y, U, V, C,
        cmap=cmap,
        angles="xy",
        scale_units="xy",
        scale=scale,
        width=0.003
    )
    ax.scatter(X, Y, c=C, cmap=cmap, s=6, alpha=0.6)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    return q


# ---------- utility --------------------------------------------------
def add_noise_along_orthogonal(x, y, dx, dy, scale=0.05):
    norm = np.hypot(dx, dy)
    ox, oy = -dy / norm, dx / norm          # unit orthogonal
    noise = np.random.normal(0, scale, size=len(x))
    return x + noise * ox, y + noise * oy

def cumulative_arc_length(x, y):
    ds = np.hypot(np.diff(x), np.diff(y))
    s  = np.insert(np.cumsum(ds), 0, 0.0)   # prepend 0
    return s / s[-1]                        # normalise 0‒1

def scale_to_unit_box(x, y):
    x_min, x_max = np.min(x), np.max(x)
    y_min, y_max = np.min(y), np.max(y)

    x_range = x_max - x_min
    y_range = y_max - y_min

    if x_range >= y_range:
        scale = 2.0 / x_range  # target: [-1, 1]
        x_scaled = (x - x_min) * scale - 1.0
        y_scaled = (y - y_min) * scale - (y_range / x_range)
    else:
        scale = 2.0 / y_range
        y_scaled = (y - y_min) * scale - 1.0
        x_scaled = (x - x_min) * scale - (x_range / y_range)

    return x_scaled, y_scaled

# ---------- simulators -----------------------------------------------
def simulate_straight_line(n=1000):
    x = np.linspace(-2, 2, n)
    y = np.zeros_like(x)
    dx, dy = np.ones_like(x), np.zeros_like(x)
    x, y   = add_noise_along_orthogonal(x, y, dx, dy)
    t      = (x + 1) / 2                    # linear time 0‒1
    return x, y, dx, dy, t

def simulate_sine_curve(n=1000, A=0.5, B=2*np.pi):
    u  = np.linspace(-2, 2, n)           # parameter along curve
    x  = u
    y  = A * np.sin(B * u)

    # analytic tangent BEFORE adding noise
    dx_raw = np.ones_like(u)
    dy_raw = A * B * np.cos(B * u)
    norm   = np.hypot(dx_raw, dy_raw)
    dx, dy = dx_raw / norm, dy_raw / norm

    # add orthogonal noise (doesn’t change tangent direction)
    x, y = add_noise_along_orthogonal(x, y, dx, dy)

    # pseudotime = arc length along noisy curve
    t = cumulative_arc_length(x, y)

    return x, y, dx, dy, t

def softplus(z, beta=6.0):
    return np.log1p(np.exp(beta * z)) / beta

def d_softplus(z, beta=6.0):
    return 1 / (1 + np.exp(-beta * z))  # sigmoid

def simulate_one_branch(
    n=800,
    t_min=-2.0,
    t_max=2.0,
    branch_time=0.0,
    slope=1.0,
    beta=6.0,
    x_shift=0.0,
    y_shift=0.0,
    base_speed=1.0,
    curvature_slowdown=0.7,
):
    t = np.linspace(t_min, t_max, n)

    z = t - branch_time

    x = t + x_shift
    y = slope * softplus(z, beta) + y_shift

    # first derivative
    dx_dt = np.ones_like(t)
    dy_dt = slope * d_softplus(z, beta)

    # second derivative: derivative of sigmoid(beta z)
    sig = d_softplus(z, beta)
    ddy_dt = slope * beta * sig * (1.0 - sig)

    # tangent direction
    tangent_norm = np.hypot(dx_dt, dy_dt) + 1e-12
    tx = dx_dt / tangent_norm
    ty = dy_dt / tangent_norm

    # approximate curvature for graph y(x)
    curvature = np.abs(ddy_dt) / (1.0 + dy_dt**2) ** 1.5

    # cars slow down in high curvature
    curvature_scaled = curvature / (np.max(curvature) + 1e-12)
    speed = base_speed * (1.0 - curvature_slowdown * curvature_scaled)

    # avoid stopping completely
    speed = np.clip(speed, 0.25 * base_speed, None)

    dx = speed * tx
    dy = speed * ty

    return x, y, dx, dy, t

def transform_branch(x, y, dx, dy, sx=1.0, sy=1.0, shift_x=0.0):
    """
    Apply symmetry:
    x -> sx * x + shift_x
    y -> sy * y

    Vector field transforms accordingly.
    """
    x_new  = sx * x + shift_x
    y_new  = sy * y
    dx_new = sx * dx
    dy_new = sy * dy
    return x_new, y_new, dx_new, dy_new

def make_branch_2(x, y, dx, dy, t, c=2.0, noise_scale=0.05):
    branches = [
        transform_branch(x, y, dx, dy, sx=+1, sy=+1, shift_x=c),
        transform_branch(x, y, dx, dy, sx=+1, sy=-1, shift_x=c),
    ]

    X, Y, DX, DY = [np.concatenate(v) for v in zip(*branches)]
    T = np.concatenate([t, t])

    if noise_scale > 0:
        X, Y = add_noise_along_orthogonal(X, Y, DX, DY, scale=noise_scale)

    return X, Y, DX, DY, T


def make_branch_4(x, y, dx, dy, t, c=2.0, noise_scale=0.05):
    branches = []

    # ---- outgoing (right) ----
    branches.append(
        transform_branch(x, y, dx, dy, sx=+1, sy=+1, shift_x=+c)
    )
    branches.append(
        transform_branch(x, y, dx, dy, sx=+1, sy=-1, shift_x=+c)
    )

    # ---- incoming (left): reverse velocity ----
    for sy in (+1, -1):
        xi, yi, dxi, dyi = transform_branch(
            x, y, dx, dy, sx=-1, sy=sy, shift_x=-c
        )
        dxi = -dxi
        dyi = -dyi
        branches.append((xi, yi, dxi, dyi))

    X, Y, DX, DY = [np.concatenate(v) for v in zip(*branches)]
    T = np.concatenate([t, t, t, t])

    if noise_scale > 0:
        X, Y = add_noise_along_orthogonal(X, Y, DX, DY, scale=noise_scale)

    return X, Y, DX, DY, T

# ---- straight & sine ------------------------------------------------
x0, y0, dx0, dy0, t0 = simulate_straight_line(n=1200)
x1, y1, dx1, dy1, t1 = simulate_sine_curve(n=1200, A=0.2, B=np.pi)

# ---- base branch primitive -----------------------------------------
x, y, dx, dy, t = simulate_one_branch(
    n=1200,
    t_min=-3, t_max=5,
    branch_time=0.0,
    slope=1.2,
    beta=4.0,
)

# ---- branch wrappers -----------------------------------------------
X2, Y2, DX2, DY2, T2 = make_branch_2(x, y, dx, dy, t, c=3.0, noise_scale=0.2)
X4, Y4, DX4, DY4, T4 = make_branch_4(x, y, dx, dy, t, c=3.0, noise_scale=0.2)

x0, y0 = scale_to_unit_box(x0, y0)
x1, y1 = scale_to_unit_box(x1, y1)
X2, Y2 = scale_to_unit_box(X2, Y2)
X4, Y4 = scale_to_unit_box(X4, Y4)


fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# ---- straight ------------------------------------------------------
q0 = plot_quiver(
    axes[0, 0],
    x0, y0, dx0, dy0,
    color=t0,
    title="Straight Line",
    scale=25
)

# ---- sine ----------------------------------------------------------
q1 = plot_quiver(
    axes[0, 1],
    x1, y1, dx1, dy1,
    color=t1,
    title="Sine Curve",
    scale=25
)

# ---- branch 2 ------------------------------------------------------
q2 = plot_quiver(
    axes[1, 0],
    X2, Y2, DX2, DY2,
    color=X2,
    title="Branch 2",
    scale=20
)

# ---- branch 4 ------------------------------------------------------
q3 = plot_quiver(
    axes[1, 1],
    X4, Y4, DX4, DY4,
    color=X4,
    title="Branch 4",
    scale=20
)

plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd

datasets = {}

# ---- straight -------------------------------------------------------
x0, y0, dx0, dy0, t0 = simulate_straight_line(n=1200)
datasets["Straight Line"] = (x0, y0, dx0, dy0, t0)

# ---- sine -----------------------------------------------------------
x1, y1, dx1, dy1, t1 = simulate_sine_curve(n=1200, A=0.2, B=np.pi)
datasets["Sine Curve"] = (x1, y1, dx1, dy1, t1)

# ---- base branch primitive -----------------------------------------
x, y, dx, dy, t = simulate_one_branch(
    n=1200,
    t_min=-3, t_max=1,
    branch_time=0.0,
    slope=1.2,
    beta=4.0,
)

# ---- branch 2 -------------------------------------------------------
X2, Y2, DX2, DY2, _ = make_branch_2(x, y, dx, dy, t, c=2.0, noise_scale=0.05)
T2 = X2.copy()   # time = x
datasets["Branch 2"] = (X2, Y2, DX2, DY2, T2)

# ---- branch 4 -------------------------------------------------------
X4, Y4, DX4, DY4, _ = make_branch_4(x, y, dx, dy, t, c=2.0, noise_scale=0.05)
T4 = X4.copy()   # time = x
datasets["Branch 4"] = (X4, Y4, DX4, DY4, T4)


# Create directory if it doesn't exist
os.makedirs("./data/1d/", exist_ok=True)

# Save each dataset
for name, (x, y, dx, dy, t) in datasets.items():
    df = pd.DataFrame({
        "x": x,
        "y": y,
        "vx": dx,
        "vy": dy,
        "time": t
    })
    # filename: lowercase, underscore
    file_name = name.lower().replace(" ", "_") + ".csv"
    file_path = os.path.join("./data/1d/", file_name)
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")